In [ ]:
!pip install -U llama-cpp-python
!pip install -q fastapi uvicorn requests

In [ ]:
# !pip install llama-cpp-python

from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="ccore/Qwen3.5-4B-3D-Coder",
    filename="Qwen3.5-4B-3D-Coder-Q4_K_M.gguf",
    n_ctx=8192,
    verbose=False
)


In [ ]:
import json
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, ConfigDict, Field, ValidationError
from typing import Any, Literal
from threading import Lock
from kaggle_secrets import UserSecretsClient
import uvicorn
import threading

SECRET_LABEL = "SYNTHETICDATA_API_TOKEN"
try:
    API_TOKEN = UserSecretsClient().get_secret(SECRET_LABEL)
except Exception as exc:
    raise RuntimeError(
        f"Missing Kaggle secret: {SECRET_LABEL}. "
        "Create it through Add-ons > Secrets and attach it to this notebook."
    ) from exc

if not API_TOKEN or not API_TOKEN.strip():
    raise RuntimeError(f"Kaggle secret {SECRET_LABEL} exists but is empty.")
API_TOKEN = API_TOKEN.strip()

app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
llm_lock = Lock()

class CapabilityItem(BaseModel):
    model_config = ConfigDict(extra="forbid")
    item_code: str
    value: float | None = None
    unit: str | None = None
    method: Literal["self_reported", "tested"] | None = None
    score: int | None = Field(default=None, ge=1, le=5)

class DemandItem(BaseModel):
    model_config = ConfigDict(extra="forbid")
    dimension_code: str
    value_state: Literal["known", "unknown"]
    value: Any = None
    unit: str | None = None
    frequency_band: Literal["never", "occasional", "frequent", "constant"] | None = None
    duration_min: float | None = Field(default=None, ge=0)
    essential: bool = False

class MatchRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    candidate_capabilities: list[CapabilityItem]
    job_demands: list[DemandItem]

class MatchResult(BaseModel):
    model_config = ConfigDict(extra="forbid")
    dimension_code: str
    status: Literal["Met", "Met with accommodation", "Gap", "Unknown"]
    candidate_item_codes: list[str]
    reason_code: str
    evidence: list[dict[str, Any]]
    confidence: Literal["high", "medium", "low"]

class MatchResponse(BaseModel):
    model_config = ConfigDict(extra="forbid")
    status: Literal["ok"]
    results: list[MatchResult]

def _match_messages(req: MatchRequest) -> list[dict[str, str]]:
    system = """You analyze only candidate capabilities against job demands.
Do not use or infer skills, disability, diagnosis, CV text, preferences,
age, gender, persona, or any other fields. Do not invent missing evidence.
Unknown evidence must remain Unknown. Return JSON only with status=ok and
one result for each job demand using the allowed statuses and confidence values."""
    user = json.dumps({
        "candidate_capabilities": [item.model_dump() for item in req.candidate_capabilities],
        "job_demands": [item.model_dump() for item in req.job_demands],
    }, ensure_ascii=False)
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def _parse_match_response(answer: str) -> MatchResponse:
    text = answer.strip()
    if text.startswith("```"):
        text = text.removeprefix("```").removeprefix("json").removesuffix("```").strip()
    return MatchResponse.model_validate(json.loads(text))

class ChatMessage(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str

class ChatRequest(BaseModel):
    messages: list[ChatMessage]
    max_tokens: int = Field(default=800, ge=1, le=2048)
    temperature: float = Field(default=0.2, ge=0, le=1)

@app.get("/health")
def health():
    return {"status": "online"}

@app.post("/chat")
def chat(req: ChatRequest, authorization: str | None = Header(default=None)):
    if authorization != f"Bearer {API_TOKEN}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    messages = [m.model_dump() for m in req.messages]
    with llm_lock:
        result = llm.create_chat_completion(messages=messages, max_tokens=req.max_tokens, temperature=req.temperature)
    return {"answer": result["choices"][0]["message"]["content"]}

@app.post("/match")
def match(req: MatchRequest, authorization: str | None = Header(default=None)):
    if authorization != f"Bearer {API_TOKEN}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    with llm_lock:
        result = llm.create_chat_completion(
            messages=_match_messages(req),
            max_tokens=1200,
            temperature=0.0,
            response_format={"type": "json_object"},
        )
    try:
        return _parse_match_response(result["choices"][0]["message"]["content"]).model_dump()
    except (KeyError, TypeError, ValueError, ValidationError) as exc:
        raise HTTPException(status_code=502, detail="Qwen returned an invalid /match response.") from exc


In [ ]:
server = uvicorn.Server(
    uvicorn.Config(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="warning"
    )
)

server_thread = threading.Thread(
    target=server.run,
    daemon=True
)

server_thread.start()

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared

!chmod +x /kaggle/working/cloudflared

In [ ]:
import subprocess
import time
import re
from pathlib import Path

log_path = "/kaggle/working/tunnel.log"

log_file = open(log_path, "w")

tunnel_process = subprocess.Popen(
    [
        "/kaggle/working/cloudflared",
        "tunnel",
        "--no-autoupdate",
        "--url",
        "http://127.0.0.1:8000"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

API_URL = None

for _ in range(60):
    log_text = Path(log_path).read_text(
        encoding="utf-8",
        errors="ignore"
    )

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        log_text
    )

    if match:
        API_URL = match.group(0)
        break

    time.sleep(1)

if API_URL is None:
    raise RuntimeError(
        "Tunnel failed. Check tunnel.log and network access."
    )

print("Your API URL:", API_URL)

In [ ]:
import requests
import time

print("FastAPI Thread:", "RUNNING" if server_thread.is_alive() else "STOPPED")
print("Cloudflare Tunnel:", "RUNNING" if tunnel_process.poll() is None else "STOPPED")

def get_public_health(url: str):
    last_error = None
    for attempt in range(10):
        try:
            return requests.get(url, timeout=20)
        except requests.RequestException as exc:
            last_error = exc
            if attempt < 9:
                time.sleep(3)
    raise RuntimeError(f"Public API did not become reachable: {last_error}") from last_error

try:
    r = requests.get("http://127.0.0.1:8000/health", timeout=5)
    print("Local API:", r.status_code, r.json())
except Exception as e:
    print("Local API Error:", repr(e))

try:
    r = get_public_health(f"{API_URL.rstrip('/')}/health")
    print("Public API:", r.status_code, r.json())
except Exception as e:
    print("Public API Error:", repr(e))

match_payload = {
    "candidate_capabilities": [{"item_code": "PC06", "value": 170, "unit": "kg", "method": "tested"}],
    "job_demands": [{"dimension_code": "D02", "value_state": "known", "value": 120, "unit": "kg", "essential": True}],
}
try:
    r = requests.post(
        f"{API_URL.rstrip('/')}/match",
        headers={"Authorization": f"Bearer {API_TOKEN}"},
        json=match_payload,
        timeout=120,
    )
    print("Match API:", r.status_code, r.json())
except Exception as e:
    print("Match API Error:", repr(e))
